In [ ]:
import pandas as pd
import numpy as np

github_raw = 'https://raw.githubusercontent.com/khoa-nhd/AIQS-Real-Estate-Price-Prediction/refs/heads/main/data/vietnam_housing_dataset.csv'

df = pd.read_csv(github_raw)

# Kiểm tra kiểu dữ liệu và số lượng non-null
df.info()

# Thống kê mô tả để tìm điểm vô lý (ví dụ: diện tích âm, số phòng = 0)
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30229 entries, 0 to 30228
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Address            30229 non-null  object 
 1   Area               30229 non-null  float64
 2   Frontage           18665 non-null  float64
 3   Access Road        16932 non-null  float64
 4   House direction    8990 non-null   object 
 5   Balcony direction  5246 non-null   object 
 6   Floors             26626 non-null  float64
 7   Bedrooms           25067 non-null  float64
 8   Bathrooms          23155 non-null  float64
 9   Legal status       25723 non-null  object 
 10  Furniture state    16110 non-null  object 
 11  Price              30229 non-null  float64
dtypes: float64(7), object(5)
memory usage: 2.8+ MB


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Price
count,30229.000000,18665.000000,16932.000000,26626.000000,25067.000000,23155.000000,30229.000000
mean,68.498741,5.361692,7.853800,3.410426,3.511030,3.346837,5.872078
std,48.069835,4.346174,7.451313,1.328897,1.309116,1.400181,2.211877
min,3.100000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,40.000000,4.000000,4.000000,2.000000,3.000000,2.000000,4.200000
50%,56.000000,4.500000,6.000000,3.000000,3.000000,3.000000,5.900000
75%,80.000000,5.000000,10.000000,4.000000,4.000000,4.000000,7.500000
max,595.000000,77.000000,85.000000,10.000000,9.000000,9.000000,11.500000


## 1. Xử lí dữ liệu trùng lặp

In [ ]:
# 1. Đếm tổng số dòng trùng lặp hoàn toàn
total_duplicates = df.duplicated().sum()
print(f"Số dòng trùng lặp 100% tất cả các cột: {total_duplicates}")

# 2. Hiển thị thử 5 cặp dữ liệu trùng lặp để quan sát bằng mắt (bắt buộc làm để hiểu rác dữ liệu từ đâu ra)
if total_duplicates > 0:
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))

# 3. Thực thi lệnh xóa (giữ lại bản ghi xuất hiện đầu tiên)
df = df.drop_duplicates(keep='first')

# 4. Kiểm tra lại kích thước dữ liệu
print(f"Kích thước dataset sau khi xóa lặp hoàn toàn: {df.shape}")

Số dòng trùng lặp 100% tất cả các cột: 0
Kích thước dataset sau khi xóa lặp hoàn toàn: (30229, 12)


## 2. Chuyển đổi định dạng dữ liệu

In [ ]:
# Chuyển đổi cột thời gian
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Chuyển đổi các biến phân loại (Categorical) để tiết kiệm bộ nhớ
categorical_cols = ['Type', 'Regionname', 'Method']
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

#3.Xử lí dữ liệu bị thiếu

In [ ]:
# Kiểm tra tỷ lệ missing
missing_ratio = df.isnull().sum() / len(df) * 100
print(missing_ratio[missing_ratio > 0])

Frontage             38.254656
Access Road          43.987562
House direction      70.260346
Balcony direction    82.645804
Floors               11.919018
Bedrooms             17.076317
Bathrooms            23.401370
Legal status         14.906216
Furniture state      46.706805
dtype: float64


#5. Kiểm soát giá trị vô lý và Ngoại lai (Outliers & Invalid Data)

In [ ]:

def cap_outliers(dataframe, column_name, lower_quantile=0.01, upper_quantile=0.99):
    """
    Hàm giới hạn ngoại lai (Capping) ở mốc 1% thấp nhất và 99% cao nhất.
    """
    # 1. Tính toán ngưỡng giới hạn
    lower_bound = dataframe[column_name].quantile(lower_quantile)
    upper_bound = dataframe[column_name].quantile(upper_quantile)

    print(f"--- Xử lý Outlier cho cột: {column_name} ---")
    print(f"Ngưỡng 1% (Lower): {lower_bound:.2f}")
    print(f"Ngưỡng 99% (Upper): {upper_bound:.2f}")

    # 2. Đếm số lượng điểm bị cắt ngọn
    outliers_upper = (dataframe[column_name] > upper_bound).sum()
    outliers_lower = (dataframe[column_name] < lower_bound).sum()
    print(f"Có {outliers_upper} điểm bị nén xuống mốc {upper_bound:.2f}")
    print(f"Có {outliers_lower} điểm bị đẩy lên mốc {lower_bound:.2f}\n")

    # 3. Thực thi Capping bằng np.clip
    dataframe[column_name] = np.clip(dataframe[column_name], lower_bound, upper_bound)

    return dataframe

# Áp dụng hàm cho cột diện tích (BuildingArea và Landsize)
# Lưu ý: Cần xử lý các giá trị 0 thành NaN trước khi capping nếu không quantile sẽ bị tính sai
if 'BuildingArea' in df.columns:
    df.loc[df['BuildingArea'] <= 0, 'BuildingArea'] = np.nan
    df = cap_outliers(df, 'BuildingArea')

# Hoàn thiện phần code bị khuất ở cuối bức ảnh của bạn
if 'Landsize' in df.columns:
    df.loc[df['Landsize'] <= 0, 'Landsize'] = np.nan
    df = cap_outliers(df, 'Landsize')

# Vẽ lại Boxplot để kiểm tra sau khi Capping (chọn BuildingArea làm đại diện)
if 'BuildingArea' in df.columns:
    plt.figure(figsize=(10, 3))
    sns.boxplot(x=df['BuildingArea'], color='lightgreen')
    plt.title('BuildingArea sau khi Capping (Ngoại lai đã được nén)')
    plt.show()

#6. Tách data address

In [ ]:

print(" BẮT ĐẦU XỬ LÝ ADDRESS VÀ FRONTAGE/ACCESS ROAD...\n")

# ---------------------------------------------------------
# BƯỚC 1: XỬ LÝ CỘT ADDRESS -> TÁCH DISTRICT & CITY
# ---------------------------------------------------------
if 'Address' in df.columns:
    # Tách chuỗi địa chỉ bằng dấu phẩy
    address_parts = df['Address'].str.split(',')

    # Lấy phần tử cuối làm City (Tỉnh/Thành), phần tử áp chót làm District (Quận/Huyện)
    # Dùng strip() để xóa khoảng trắng thừa ở hai đầu chữ
    df['City'] = address_parts.apply(lambda x: x[-1].strip() if isinstance(x, list) and len(x) >= 1 else np.nan)
    df['District'] = address_parts.apply(lambda x: x[-2].strip() if isinstance(x, list) and len(x) >= 2 else np.nan)

    # Xóa cột Address gốc để giảm rác mô hình
    df = df.drop(columns=['Address'])
    print("✔️ Đã tách thành công cột 'Address' thành 'District' và 'City'.")
else:
    print("⚠️ Cột 'Address' không tồn tại hoặc đã được xử lý.")

# ---------------------------------------------------------
# BƯỚC 2: XỬ LÝ MISSING VALUES CHO FRONTAGE & ACCESS ROAD
# ---------------------------------------------------------
# Tính tỷ lệ missing trước khi xử lý
missing_frontage_before = df['Frontage'].isnull().sum() if 'Frontage' in df.columns else 0
missing_access_before = df['Access Road'].isnull().sum() if 'Access Road' in df.columns else 0

if missing_frontage_before > 0 or missing_access_before > 0:
    # 2.1. Impute bằng Median theo từng Quận/Huyện (District)
    if 'District' in df.columns:
        if 'Frontage' in df.columns:
            df['Frontage'] = df.groupby('District')['Frontage'].transform(lambda x: x.fillna(x.median()))
        if 'Access Road' in df.columns:
            df['Access Road'] = df.groupby('District')['Access Road'].transform(lambda x: x.fillna(x.median()))
        print("✔️ Đã điền khuyết Frontage/Access Road bằng Median của từng District.")

    # 2.2. Fallback: Điền các ô vẫn còn NaN (Do District đó cũng NaN toàn bộ)
    # Logic: Frontage NaN -> Mặt tiền = 0 (nhà hẻm). Access Road NaN -> Đường vào = 2m
    if 'Frontage' in df.columns:
        df['Frontage'] = df['Frontage'].fillna(0)
    if 'Access Road' in df.columns:
        df['Access Road'] = df['Access Road'].fillna(2.0)

    print("✔️ Đã áp dụng Fallback (Frontage=0, Access Road=2.0) cho các trường hợp ngoại lệ.")

# ---------------------------------------------------------
# BƯỚC 3: KIỂM TRA LẠI KẾT QUẢ
# ---------------------------------------------------------
print("\n--- THỐNG KÊ SAU KHI XỬ LÝ ---")
cols_to_check = ['District', 'City', 'Frontage', 'Access Road']
existing_cols = [col for col in cols_to_check if col in df.columns]

print(f"Số lượng NaN hiện tại của các cột:")
print(df[existing_cols].isnull().sum())

print("\n--- 5 DÒNG DỮ LIỆU MẪU ---")
display(df[existing_cols].head())

 BẮT ĐẦU XỬ LÝ ADDRESS VÀ FRONTAGE/ACCESS ROAD...

✔️ Đã tách thành công cột 'Address' thành 'District' và 'City'.
✔️ Đã điền khuyết Frontage/Access Road bằng Median của từng District.
✔️ Đã áp dụng Fallback (Frontage=0, Access Road=2.0) cho các trường hợp ngoại lệ.

--- THỐNG KÊ SAU KHI XỬ LÝ ---
Số lượng NaN hiện tại của các cột:
District       3
City           0
Frontage       0
Access Road    0
dtype: int64

--- 5 DÒNG DỮ LIỆU MẪU ---


,District,City,Frontage,Access Road
0,Văn Giang,Hưng Yên,5.0,13.0
1,Văn Giang,Hưng Yên,5.0,13.0
2,Văn Giang,Hưng Yên,6.0,13.0
3,Gò Vấp,Hồ Chí Minh,4.1,3.5
4,Gò Vấp,Hồ Chí Minh,4.1,5.0


#7. Final check


In [ ]:
# 1. Kiểm tra Dữ liệu khuyết thiếu (Missing Values)
# Trả về True nếu không còn bất kỳ ô NaN nào trong toàn bộ dataset
print("1. Đã sạch missing values hoàn toàn chưa?:", df.isnull().sum().max() == 0)

# Nếu bạn chủ đích giữ lại NaN ở một số cột, phải kiểm tra cụ thể cột mục tiêu (vd: Price)
print("1b. Cột Price đã sạch NaN chưa?:", df['Price'].isnull().sum() == 0)

# 2. Kiểm tra Dữ liệu trùng lặp (Duplicates)
# Trả về True nếu không còn dòng nào giống nhau 100%
print("2. Đã sạch dòng trùng lặp chưa?:", df.duplicated().sum() == 0)

# 4. Kiểm tra cấu trúc kiểu dữ liệu (Data Types)
# Biến phân loại (Categorical) không được lưu dưới dạng chuỗi rác (Object) hoặc float
print("\n4. Kiểm tra DataType hiện tại:")
df.info()

1. Đã sạch missing values hoàn toàn chưa?: False
1b. Cột Price đã sạch NaN chưa?: True
2. Đã sạch dòng trùng lặp chưa?: False

4. Kiểm tra DataType hiện tại:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30229 entries, 0 to 30228
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Area               30229 non-null  float64
 1   Frontage           30229 non-null  float64
 2   Access Road        30229 non-null  float64
 3   House direction    8990 non-null   object 
 4   Balcony direction  5246 non-null   object 
 5   Floors             26626 non-null  float64
 6   Bedrooms           25067 non-null  float64
 7   Bathrooms          23155 non-null  float64
 8   Legal status       25723 non-null  object 
 9   Furniture state    16110 non-null  object 
 10  Price              30229 non-null  float64
 11  City               30229 non-null  object 
 12  District           30226 non-null  object 
dtypes: float

#8. Download Data

In [ ]:
from google.colab import files

# 1. Đặt tên cho file dữ liệu đã làm sạch
clean_file_name = "data_estate_house_VN.csv"

# 2. Xuất DataFrame ra file CSV
# LỖ HỔNG CẦN TRÁNH: Bắt buộc phải có index=False.
# Nếu không, Pandas sẽ tự động sinh ra một cột số thứ tự vô nghĩa ở đầu file, làm hỏng cấu trúc ma trận X khi train model.
# encoding='utf-8-sig' đảm bảo font tiếng Việt (nếu có) không bị lỗi font khi người khác mở bằng Excel.
df.to_csv(clean_file_name, index=False, encoding='utf-8-sig')

# 3. Kích hoạt lệnh tải file về máy tính cục bộ
files.download(clean_file_name)
print(f"✅ Đã xuất thành công file {clean_file_name} và đang tải về máy!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Đã xuất thành công file data_estate_house_VN.csv và đang tải về máy!
